In [ ]:
# !pip3 install -U --force-reinstall --no-cache-dir \
#     torch==2.6.0 \
#     transformers \
#     datasets \
#     vllm \
#     pandas \
#     tqdm \
#     scikit-learn

: 

In [ ]:
# !pip3 install "typer<0.10.0,>=0.3.0"

  Using cached typer-0.9.4-py3-none-any.whl.metadata (14 kB)
Using cached typer-0.9.4-py3-none-any.whl (45 kB)
  Attempting uninstall: typer
    Found existing installation: typer 0.23.2
    Uninstalling typer-0.23.2:
      Successfully uninstalled typer-0.23.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi-cli 0.0.22 requires typer>=0.16.0, but you have typer 0.9.4 which is incompatible.
fastapi-cloud-cli 0.12.0 requires typer>=0.16.0, but you have typer 0.9.4 which is incompatible.


In [1]:
import datetime
import json
import os
import re
import time
from typing import List

import torch
from datasets import load_dataset
from sklearn.metrics import classification_report
from tqdm import tqdm
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


/Users/zhusizhen/Documents/project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
import pandas as pd

In [4]:
# pip install --upgrade huggingface_hub

In [3]:
from huggingface_hub import hf_hub_download

In [5]:
# hf_hub_download(repo_id="google/pegasus-xsum", filename="config.json")

In [6]:
# %pip install --upgrade --force-reinstall "huggingface_hub>=0.24.0" transformers datasets

In [7]:
# %pip install --upgrade "typer>=0.16.0"

In [4]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")

### loading the data

In [5]:
from datasets import load_dataset

In [6]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("seyled/Phantom_Hallucination_Detection")

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

In [ ]:
# TMP_DIR = "/tmp/"

# data_names = ['10k_seed', '8k_seed', '497k_seed', 'def14a_seed',
#               '10k_20000tokens_beginning', '10k_20000tokens_end', '10k_20000tokens_middle',
#               '10k_2000tokens_beginning', '10k_2000tokens_end', '10k_2000tokens_middle',
#               '50000tokens_beginning', '50000tokens_end', '50000tokens_middle',
#               '10k_5000tokens_beginning', '10k_5000tokens_end', '10k_5000tokens_middle',
#               '10k_30000tokens_end','10k_30000tokens_beginning','10k_30000tokens_middle',
#               '10k_10000tokens_beginning', '10k_10000tokens_end', '10k_10000tokens_middle'
#               'def14A_10000tokens_beginning', 'def14A_10000tokens_end', 'def14A_10000tokens_middle'
#               'def14A_2000tokens_middle','def14A_2000tokens_beginning','def14A_2000tokens_end',
#               'def14A_5000tokens_middle','def14A_5000tokens_beginning','def14A_5000tokens_end',
#               'def14A_20000tokens_middle','def14A_20000tokens_end','def14A_20000tokens_beginning',
#               'def14A_30000tokens_end','def14A_30000tokens_beginning','def14A_30000tokens_middle',
#               'def14A_10000tokens_beginning', 'def14A_10000tokens_end', 'def14A_10000tokens_middle'

# ]


# detector_model_list = ['deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-14B', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B',
#                        "Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-32B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.3-70B-Instruct",
#                        "deepseek-ai/DeepSeek-R1-Distill-Llama-70B"]

In [7]:
df = load_dataset("seyled/Phantom_Hallucination_Detection", data_files="PhantomDataset/Phantom_10k_seed.csv")

In [8]:
df_trans = pd.DataFrame(df["train"])

In [9]:
df_trans.head()

,Unnamed: 0,query,context,answer,ground_truth_label
0,0,What are some potential macroeconomic risks th...,Water is an essential component of our drillin...,Macroeconomic risks that could have an adverse...,not hallucination
1,1,What are some potential macroeconomic risks th...,Water is an essential component of our drillin...,Macroeconomic risks that could have an adverse...,hallucination
2,2,What percentage of the company's net revenue w...,Some of our competitors may possess greater re...,41%,not hallucination
3,3,What percentage of the company's net revenue w...,Some of our competitors may possess greater re...,"In 2023, the company generated approximately 4...",hallucination
4,4,What is the company's policy regarding the use...,Natural gas and NGLs are stored in large volum...,The company's policy is not to acquire and hol...,not hallucination


In [10]:
df_trans.shape

(994, 5)

### Define and Create the Prompt for the Hallucination Detector Model

In [11]:
"""
Put your prompt for hallucination detection in the function below (extend HALLUCINATION_PROMPT string).

We follow these guidelines when creating the hallucination detection prompt:
	1. Detect any deviation in the answer from the context.
	2. Detect implied or implicit information in answer that is not present in the context.
	3. Detect misalignment of timing in the answer that might be different from the context.
	4. Detect any important details in the context that is missed in the answer.
	5. Ask the model to provide reasoning (i.e., chain of thought).
	6. Ask for pass/fail decision, depending on whether the models finds the answer faithful to the context or not.
"""
def create_detection_prompt(query, chunk, answer):
        HALLUCINATION_PROMPT = f"""
        **Input Format:**
        --
        QUESTION:
        {query}
        CONTEXT:
        {chunk}
        ANSWER:
        {answer}
        --

        **Example output:**
        {{"REASONING": [...], "SCORE": "PASS" or "FAIL"}}
        """
        return HALLUCINATION_PROMPT

In [20]:
class VllmModel:
    """
    A lightweight fallback class that uses small, fast generation settings.
    """

    def __init__(self, model_path: str = "Qwen/Qwen2.5-7B-Instruct", tensor_parallel_size: int = 2,
                max_token_length=32000):
        self.model_path = model_path
        self.max_token_length = max_token_length

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

        device = "mps" if torch.backends.mps.is_available() else "cpu"
        if device == "mps":
            dtype = torch.float16
            load_in_8bit = False
            device_map = None
        else:
            dtype = torch.float32
            load_in_8bit = False
            device_map = "cpu"

        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=dtype,
            load_in_8bit=load_in_8bit,
            device_map=device_map,
            low_cpu_mem_usage=True,
        )

        if device == "mps":
            self.model = self.model.to("cpu")

        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device=-1,
        )

    def generate_prompts(self, prompts: List[str]):
        model_input = []
        for prompt in tqdm(prompts, desc="Generating outputs"):
            messages = [{"role": "user", "content": prompt}]
            chat_text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            encoded_input = self.tokenizer.encode(chat_text, truncation=False)

            if len(encoded_input) > self.max_token_length:
                truncated_input = encoded_input[: self.max_token_length]
                chat_text = self.tokenizer.decode(truncated_input, skip_special_tokens=True)
            model_input.append(chat_text)

        outputs = self.generator(
            model_input,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        return outputs

    def generate_responses(self, prompts: List[str], return_answer_only: bool = True):
        outputs = self.generate_prompts(prompts)
        if return_answer_only:
            outputs = [output[0]["generated_text"] for output in outputs]
        return outputs


In [13]:
re_map = {
    "pass": "not hallucination",
    "fail": "hallucination"
}

def get_label(response):
    """
    Parses a string containing JSON-like model response to extract the 'SCORE' and determine if it indicates a hallucination.

    Args:
        response: A string containing JSON-like data.

    Returns:
        "not hallucination" if the SCORE is "PASS", "hallucination" if the SCORE is "FAIL".
        Returns None if the SCORE cannot be determined.
    """
    try:
        # Attempt to load the response as JSON
        j_res = json.loads(response).lower()
        score = j_res.get("score").strip()
        if score in re_map:
            return re_map[score]
        else:
            # throw exception
            raise ValueError(f"Unexpected score value: {score}")

    except:
        # If JSON parsing fails, use regular expressions to find the SCORE
        try:
            match = re.search(r'"SCORE"\s*:\s*"(\w+)"', response, re.IGNORECASE)
            if match:
                score = match.group(1).lower().strip()
                if score in re_map:
                    return re_map[score]
                else:
                    raise ValueError(f"Unexpected score value: {score}")
            else:
                if '"FAIL"' in response:
                    return 'hallucination'
                else:
                    return 'not hallucination'
        except:
            return 'not hallucination'

### Testing

In [14]:
detector_model_name = 'Qwen/Qwen2.5-1.5B-Instruct'


In [15]:
data_name = '10k_seed'

In [16]:
path = "/Users/zhusizhen/Documents/project/data"
output_output_file_name = f'{path}/outputs_{detector_model_name.replace("/", "__")}_{data_name}.csv'

In [17]:
model = VllmModel(detector_model_name, tensor_parallel_size=4, max_token_length=32000)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cpu


In [18]:
df_final = df["train"].to_pandas()

In [19]:
df_final[f'{detector_model_name}_output'] = None

In [36]:
start_time = time.time()
prompt_list = []

# Use a tiny subset first for smoke testing
sample_df = df_final.head(1).copy()
for i in range(sample_df.shape[0]):
    query = sample_df.at[i, 'query']
    chunk = sample_df.at[i, 'context']
    answer = sample_df.at[i, 'answer']

    HALLUCINATION_PROMPT = create_detection_prompt(query, chunk, answer)
    prompt_list.append(HALLUCINATION_PROMPT)

model_output_list = model.generate_responses(prompt_list)
sample_df[f'{detector_model_name}_output'] = model_output_list
sample_df.to_csv(output_output_file_name, index=False)
print('Smoketest completed for', sample_df.shape[0], 'rows')
print('Saved to', output_output_file_name)
print('Process time:', round(time.time() - start_time, 2), 'seconds.')


Generating outputs:   0%|          | 0/1 [00:00<?, ?it/s]

Smoketest completed for 1 rows
Saved to /Users/zhusizhen/Documents/project/data/outputs_Qwen__Qwen2.5-1.5B-Instruct_10k_seed.csv
Process time: 55.82 seconds.


In [37]:
df = pd.read_csv("/Users/zhusizhen/Documents/project/data/outputs_Qwen__Qwen2.5-1.5B-Instruct_10k_seed.csv")
df.head()

,Unnamed: 0,query,context,answer,ground_truth_label,Qwen/Qwen2.5-1.5B-Instruct_output,Qwen/Qwen2.5-1.5B-Instruct_label
0,0,What are some potential macroeconomic risks th...,Water is an essential component of our drillin...,Macroeconomic risks that could have an adverse...,not hallucination,"<|im_start|>system\nYou are Qwen, created by A...",not hallucination


In [38]:
df["Qwen/Qwen2.5-1.5B-Instruct_output"].unique()

array(['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n\n        **Input Format:**\n        --\n        QUESTION:\n        What are some potential macroeconomic risks that could have an adverse impact on the company\'s business and financial results?\n        CONTEXT:\n        Water is an essential component of our drilling and hydraulic fracturing processes. If we are unable to obtain water to use in our operations from local sources, we may be unable to economically produce oil, natural gas liquids and natural gas, which could have an adverse effect on our business, financial condition and results of operations. Wastewaters from our operations typically are disposed of via underground injection. Some studies have linked earthquakes in certain areas to underground injection, which is leading to greater public scrutiny and regulation of disposal wells. Any new environmental initiatives or regulations that restrict i

In [41]:
print('\nStart parsing ', detector_model_name, ' results.')

sample_df[f'{detector_model_name}_label'] = None

for i in range(df.shape[0]):
    df.at[i, f'{detector_model_name}_label'] = get_label(df.at[i, f'{detector_model_name}_output'])
    print('\nFinished parsing ', detector_model_name)



Start parsing  Qwen/Qwen2.5-1.5B-Instruct  results.

Finished parsing  Qwen/Qwen2.5-1.5B-Instruct


In [42]:
df.head()

,Unnamed: 0,query,context,answer,ground_truth_label,Qwen/Qwen2.5-1.5B-Instruct_output,Qwen/Qwen2.5-1.5B-Instruct_label
0,0,What are some potential macroeconomic risks th...,Water is an essential component of our drillin...,Macroeconomic risks that could have an adverse...,not hallucination,"<|im_start|>system\nYou are Qwen, created by A...",not hallucination


In [46]:
df.shape

(1, 7)

In [47]:
df["Qwen/Qwen2.5-1.5B-Instruct_label"].unique()

array(['not hallucination'], dtype=object)

In [48]:
## OUTPUT
with open(output_output_file_name, 'w') as f:
    f.write(f'Phantom {data_name} results:')
    print(f'Phantom {data_name} results:')

    results = classification_report(df['ground_truth_label'], df[f'{detector_model_name}_label'],
                                            digits=3)
    f.write(f'\n\n{detector_model_name} results.')
    f.write(results)
    print(f'\n\n{detector_model_name} results.')
    print(results)

Phantom 10k_seed results:


Qwen/Qwen2.5-1.5B-Instruct results.
                   precision    recall  f1-score   support

not hallucination      1.000     1.000     1.000         1

         accuracy                          1.000         1
        macro avg      1.000     1.000     1.000         1
     weighted avg      1.000     1.000     1.000         1

